# Import Libraries

d) Compare the performance of different models and strategies. (20 points)
1. Use at least three transfer models that you've trained using task(b) and perform ensemble
learning. Try out the following ensemble techniques (Stacking, Boosting, Weighted Average,
Max Voting, Bagging) and analyze whether the performance increases or not.  
2. Try out different image preprocessing techniques such as, Ben Graham, Circle Cropping,
CLAHE, adding gaussian blur, sharpening up the images etc.

In [21]:
import os
import sys
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, accuracy_score
from torch.utils.data import Dataset
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

In [22]:
# Set seeds for Python's `random`
random.seed(42)

# Set seeds for NumPy
np.random.seed(42)

# Set seeds for PyTorch (if PyHRPN is dependent on PyTorch)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [23]:
class RetinopathyDataset(Dataset):
    def __init__(self, ann_file, image_dir, transform=None, mode='single', test=False):
        self.ann_file = ann_file
        self.image_dir = image_dir
        self.transform = transform

        self.test = test
        self.mode = mode

        if self.mode == 'single':
            self.data = self.load_data()
        else:
            self.data = self.load_dual_data()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        if self.mode == 'single':
            return self.get_item(index)
        else:
            return self.get_item_dual(index)

    # 1. single image
    def load_data(self):
        df = pd.read_csv(self.ann_file)

        data = []
        for _, row in df.iterrows():
            file_info = dict()
            file_info['img_path'] = os.path.join(self.image_dir, row['img_path'])
            if not self.test:
                file_info['dr_level'] = int(row['patient_DR_Level'])
            data.append(file_info)
        return data

    def get_item(self, index):
        data = self.data[index]
        img = Image.open(data['img_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)

        if not self.test:
            label = torch.tensor(data['dr_level'], dtype=torch.int64)
            return img, label
        else:
            return img

    # 2. dual image
    def load_dual_data(self):
        df = pd.read_csv(self.ann_file)

        df['prefix'] = df['image_id'].str.split('_').str[0]  # The patient id of each image
        df['suffix'] = df['image_id'].str.split('_').str[1].str[0]  # The left or right eye
        grouped = df.groupby(['prefix', 'suffix'])

        data = []
        for (prefix, suffix), group in grouped:
            file_info = dict()
            file_info['img_path1'] = os.path.join(self.image_dir, group.iloc[0]['img_path'])
            file_info['img_path2'] = os.path.join(self.image_dir, group.iloc[1]['img_path'])
            if not self.test:
                file_info['dr_level'] = int(group.iloc[0]['patient_DR_Level'])
            data.append(file_info)
        return data

    def get_item_dual(self, index):
        data = self.data[index]
        img1 = Image.open(data['img_path1']).convert('RGB')
        img2 = Image.open(data['img_path2']).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        if not self.test:
            label = torch.tensor(data['dr_level'], dtype=torch.int64)
            return [img1, img2], label
        else:
            return [img1, img2]



def train_model(model, train_loader, val_loader, device, criterion, optimizer, lr_scheduler, num_epochs=25,
                checkpoint_path='model.pth'):
    best_model = model.state_dict()
    best_epoch = None
    best_val_kappa = -1.0  # Initialize the best kappa score

    # Initialize metrics
    training_loss = []
    validation_loss = []
    training_accuracy = []
    validation_accuracy = []
    train_kappa = []
    val_kappa_list = []

    for epoch in range(1, num_epochs + 1):
        print(f'\nEpoch {epoch}/{num_epochs}')
        running_loss = []
        all_preds = []
        all_labels = []

        model.train()

        with tqdm(total=len(train_loader), desc=f'Training', unit=' batch', file=sys.stdout) as pbar:
            for images, labels in train_loader:
                if not isinstance(images, list):
                    images = images.to(device)  # single image case
                else:
                    images = [x.to(device) for x in images]  # dual images case

                labels = labels.to(device)

                optimizer.zero_grad()

                outputs = model(images)
                loss = criterion(outputs, labels.long())

                loss.backward()
                optimizer.step()

                preds = torch.argmax(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                running_loss.append(loss.item())

                pbar.set_postfix({'lr': f'{optimizer.param_groups[0]["lr"]:.1e}', 'Loss': f'{loss.item():.4f}'})
                pbar.update(1)

        lr_scheduler.step()

        epoch_loss = sum(running_loss) / len(running_loss)

        train_metrics = compute_metrics(all_preds, all_labels, per_class=True)
        kappa, accuracy, precision, recall = train_metrics[:4]

        print(f'[Train] Kappa: {kappa:.4f} Accuracy: {accuracy:.4f} '
              f'Precision: {precision:.4f} Recall: {recall:.4f} Loss: {epoch_loss:.4f}')

        if len(train_metrics) > 4:
            precision_per_class, recall_per_class = train_metrics[4:]
            for i, (precision, recall) in enumerate(zip(precision_per_class, recall_per_class)):
                print(f'[Train] Class {i}: Precision: {precision:.4f}, Recall: {recall:.4f}')

        # Evaluation on the validation set at the end of each epoch
        val_metrics, val_loss= evaluate_model(criterion, model, val_loader, device)
        val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
        print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')

        if val_kappa > best_val_kappa:
            best_val_kappa = val_kappa
            best_epoch = epoch
            best_model = model.state_dict()
            torch.save(best_model, checkpoint_path)

        training_loss.append(epoch_loss)
        validation_loss.append(val_loss)
        training_accuracy.append(accuracy)
        validation_accuracy.append(val_accuracy)
        train_kappa.append(kappa)
        val_kappa_list.append(val_kappa)

    print(f'[Val] Best kappa: {best_val_kappa:.4f}, Epoch {best_epoch}')

    return model, training_loss, validation_loss, training_accuracy, validation_accuracy, train_kappa, val_kappa_list


def evaluate_model(criterion, model, test_loader, device, test_only=False, prediction_path='./test_predictions.csv'):
    model.eval()

    all_preds = []
    all_labels = []
    all_image_ids = []

    with tqdm(total=len(test_loader), desc=f'Evaluating', unit=' batch', file=sys.stdout) as pbar:
        for i, data in enumerate(test_loader):

            if test_only:
                images = data
            else:
                images, labels = data

            if not isinstance(images, list):
                images = images.to(device)  # single image case
            else:
                images = [x.to(device) for x in images]  # dual images case

            with torch.no_grad():
                outputs = model(images)
                preds = torch.argmax(outputs, 1)
            if not test_only:
                val_loss = criterion(outputs, labels.to(device).long())
                val_loss = val_loss.cpu().numpy()

            if not isinstance(images, list):
                # single image case
                all_preds.extend(preds.cpu().numpy())
                image_ids = [
                    os.path.basename(test_loader.dataset.data[idx]['img_path']) for idx in
                    range(i * test_loader.batch_size, i * test_loader.batch_size + len(images))
                ]
                all_image_ids.extend(image_ids)
                if not test_only:
                    all_labels.extend(labels.numpy())
            else:
                # dual images case
                for k in range(2):
                    all_preds.extend(preds.cpu().numpy())
                    image_ids = [
                        os.path.basename(test_loader.dataset.data[idx][f'img_path{k + 1}']) for idx in
                        range(i * test_loader.batch_size, i * test_loader.batch_size + len(images[k]))
                    ]
                    all_image_ids.extend(image_ids)
                    if not test_only:
                        all_labels.extend(labels.numpy())

            pbar.update(1)

    # Save predictions to csv file for Kaggle online evaluation
    if test_only:
        df = pd.DataFrame({
            'ID': all_image_ids,
            'TARGET': all_preds
        })
        df.to_csv(prediction_path, index=False)
        print(f'[Test] Save predictions to {os.path.abspath(prediction_path)}')
    else:
        metrics = compute_metrics(all_preds, all_labels)
        return metrics, val_loss

def compute_metrics(preds, labels, per_class=False):
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='weighted', zero_division=0)
    recall = recall_score(labels, preds, average='weighted', zero_division=0)

    # Calculate and print precision and recall for each class
    if per_class:
        precision_per_class = precision_score(labels, preds, average=None, zero_division=0)
        recall_per_class = recall_score(labels, preds, average=None, zero_division=0)
        return kappa, accuracy, precision, recall, precision_per_class, recall_per_class

    return kappa, accuracy, precision, recall

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Augmentation Functions

In [25]:
class SLORandomPad:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        pad_width = max(0, self.size[0] - img.width)
        pad_height = max(0, self.size[1] - img.height)
        pad_left = random.randint(0, pad_width)
        pad_top = random.randint(0, pad_height)
        pad_right = pad_width - pad_left
        pad_bottom = pad_height - pad_top
        return transforms.functional.pad(img, (pad_left, pad_top, pad_right, pad_bottom))


class FundRandomRotate:
    def __init__(self, prob, degree):
        self.prob = prob
        self.degree = degree

    def __call__(self, img):
        if random.random() < self.prob:
            angle = random.uniform(-self.degree, self.degree)
            return transforms.functional.rotate(img, angle)
        return img

# Hyper Parameters

In [26]:
# Hyper Parameters
batch_size = 48
num_classes = 5  # 5 DR levels
learning_rate = 0.0001
num_epochs = 20

# Transformers

In [27]:
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((210, 210)),
    SLORandomPad((224, 224)),
    FundRandomRotate(prob=0.5, degree=30),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=(0.1, 0.9)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Main Loop

In [28]:
# Define the weighted CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

# Use GPU device is possible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


# Single Models

In [29]:
# Choose between 'single image' and 'dual images' pipeline
# This will affect the model definition, dataset pipeline, training and evaluation

mode = 'single'  # forward single image to the model each time

In [30]:
# Create datasets
train_dataset = RetinopathyDataset('/content/drive/MyDrive/DL_Project/DeepDRiD/train.csv', '/content/drive/MyDrive/DL_Project/DeepDRiD/train/', transform_train, mode)
val_dataset = RetinopathyDataset('/content/drive/MyDrive/DL_Project/DeepDRiD/val.csv', '/content/drive/MyDrive/DL_Project/DeepDRiD/val/', transform_test, mode)
test_dataset = RetinopathyDataset('/content/drive/MyDrive/DL_Project/DeepDRiD/test.csv', '/content/drive/MyDrive/DL_Project/DeepDRiD//test/', transform_test, mode, test=True)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [31]:
class LoadModels(nn.Module):
    def __init__(self, model_name, num_classes=5):
        super(LoadModels, self).__init__()
        self.model_name = model_name
        if self.model_name == 'resnet18':
            self.backbone = models.resnet18(pretrained=False)
            num_ftrs = self.backbone.fc.in_features
            self.backbone.fc = nn.Linear(num_ftrs, num_classes)
            state_dict = torch.load('/content/drive/MyDrive/DL_Project/part_b/resnet18.pth', map_location='cpu')
        elif self.model_name == 'resnet34':
            self.backbone = models.resnet34(pretrained=False)
            num_ftrs = self.backbone.fc.in_features
            self.backbone.fc = nn.Linear(num_ftrs, num_classes)
            state_dict = torch.load('/content/drive/MyDrive/DL_Project/part_b/resnet34.pth', map_location='cpu')
        elif self.model_name == 'vgg16':
            self.backbone = models.vgg16(pretrained=False)
            num_ftrs = self.backbone.classifier[6].in_features
            self.backbone.classifier[6] = nn.Linear(num_ftrs, num_classes)
            state_dict = torch.load('/content/drive/MyDrive/DL_Project/part_b/vgg16.pth', map_location='cpu')
        elif self.model_name == 'efficientnet_b0':
            self.backbone = models.efficientnet_b0(pretrained=False)
            num_ftrs = self.backbone.classifier[1].in_features
            self.backbone.classifier[1] = nn.Linear(num_ftrs, num_classes)
            state_dict = torch.load('/content/drive/MyDrive/DL_Project/part_b/efficientnet_b0.pth', map_location='cpu')
        elif self.model_name == 'densenet121':
            self.backbone = models.densenet121(pretrained=False)
            num_ftrs = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Linear(num_ftrs, num_classes)
            state_dict = torch.load('/content/drive/MyDrive/DL_Project/part_b/densenet121.pth', map_location='cpu')

        # Remove 'backbone.' prefix from keys
        new_state_dict = {}
        for k, v in state_dict.items():
            if k.startswith('backbone.'):
                new_state_dict[k[len('backbone.'):]] = v
            else:
                new_state_dict[k] = v
        self.backbone.load_state_dict(new_state_dict, strict=True)

    def forward(self, x):
        return self.backbone(x)

In [32]:
resnet18 = LoadModels('resnet18')
resnet34 = LoadModels('resnet34')
vgg16 = LoadModels('vgg16')
efficientnet_b0 = LoadModels('efficientnet_b0')
densenet121 = LoadModels('densenet121')

models = [resnet18, resnet34, vgg16, efficientnet_b0, densenet121]
[model.to(device) for model in models]
# Switch all models to evaluation mode
[model.eval() for model in models]

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-31-a339cd54a3ff>:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions

[LoadModels(
   (backbone): ResNet(
     (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
     (layer1): Sequential(
       (0): BasicBlock(
         (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
         (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
         (relu): ReLU(inplace=True)
         (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
         (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       )
       (1): BasicBlock(
         (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
         (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, a

Test Models independently anc Compute Weights

In [33]:
val_metrics, val_loss= evaluate_model(criterion, resnet18, val_loader, device)
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print("ResNet18 - Metrics:")
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')
val_metrics, val_loss= evaluate_model(criterion, resnet34, val_loader, device)
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print("ResNet34 - Metrics:")
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')
val_metrics, val_loss= evaluate_model(criterion, vgg16, val_loader, device)
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print("VGG16 - Metrics:")
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')
val_metrics, val_loss= evaluate_model(criterion, efficientnet_b0, val_loader, device)
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print("EfficientNet B0 - Metrics:")
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')
val_metrics, val_loss= evaluate_model(criterion, densenet121, val_loader, device)
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print("DenseNet121 - Metrics:")
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')


def compute_weights_from_kappa(kappa_scores):

    total_kappa = sum(kappa_scores.values())

    # Normalize each kappa score
    weights = [kappa / total_kappa for model, kappa in kappa_scores.items()]
    return weights


# Kappa scores from validation metrics
kappa_scores = {
    'ResNet18': 0.8542,
    'ResNet34': 0.8554,
    'VGG16': 0.8337,
    'EfficientNet_B0': 0.8318,
    'DenseNet121': 0.8639
}

# Compute weights
weights = compute_weights_from_kappa(kappa_scores)
weights

Evaluating: 100%|██████████| 9/9 [00:03<00:00,  2.31 batch/s]
ResNet18 - Metrics:
[Val] Kappa: 0.8542 Accuracy: 0.6875 Precision: 0.6709 Recall: 0.6875
Evaluating: 100%|██████████| 9/9 [00:04<00:00,  2.21 batch/s]
ResNet34 - Metrics:
[Val] Kappa: 0.8554 Accuracy: 0.6950 Precision: 0.6798 Recall: 0.6950
Evaluating: 100%|██████████| 9/9 [00:06<00:00,  1.49 batch/s]
VGG16 - Metrics:
[Val] Kappa: 0.8337 Accuracy: 0.6950 Precision: 0.6806 Recall: 0.6950
Evaluating: 100%|██████████| 9/9 [00:03<00:00,  2.34 batch/s]
EfficientNet B0 - Metrics:
[Val] Kappa: 0.8318 Accuracy: 0.7125 Precision: 0.7403 Recall: 0.7125
Evaluating: 100%|██████████| 9/9 [00:04<00:00,  1.97 batch/s]
DenseNet121 - Metrics:
[Val] Kappa: 0.8639 Accuracy: 0.7025 Precision: 0.6949 Recall: 0.7025


[0.2015097900448219,
 0.201792875678226,
 0.19667374380750177,
 0.19622552488794529,
 0.20379806558150507]

# Weighted Average, Max Voting

In [34]:
def weighted_average_ensemble(models, inputs, weights):
    outputs = [model(inputs) * weight for model, weight in zip(models, weights)]
    avg_output = torch.stack(outputs).sum(dim=0)
    return avg_output

def max_voting_ensemble(models, inputs):
    outputs = [model(inputs) for model in models]
    votes = [torch.argmax(output, dim=1) for output in outputs]
    votes = torch.stack(votes, dim=1)
    majority_vote, _ = torch.mode(votes, dim=1)
    return majority_vote

def bagging_ensemble(models, inputs):
    outputs = [model(inputs) for model in models]
    avg_output = torch.stack(outputs).mean(dim=0)
    return avg_output

def evaluate_ensemble(criterion, models, val_loader, device, ensemble_method, weights=None):

    all_preds = []
    all_labels = []
    all_image_ids = []

    with tqdm(total=len(val_loader), desc=f'Evaluating', unit=' batch', file=sys.stdout) as pbar:
        for i, data in enumerate(val_loader):
            images, labels = data

            if not isinstance(images, list):
                images = images.to(device)  # single image case
            else:
                images = [x.to(device) for x in images]  # dual images case

            with torch.no_grad():
                if ensemble_method == 'weighted_average':
                    outputs = weighted_average_ensemble(models, images, weights)
                    preds = torch.argmax(outputs, dim=1)
                elif ensemble_method == 'max_voting':
                    preds = max_voting_ensemble(models, images)
                    outputs = torch.stack([model(images) for model in models]).mean(dim=0)  # Dummy output for loss calculation
                elif ensemble_method == 'bagging':
                    outputs = bagging_ensemble(models, images)
                    preds = torch.argmax(outputs, dim=1)
                else:
                    raise ValueError("Invalid ensemble method")

                val_loss = criterion(outputs, labels.to(device).long())
                val_loss = val_loss.cpu().numpy()

            if not isinstance(images, list):
                # single image case
                all_preds.extend(preds.cpu().numpy())
                image_ids = [
                    os.path.basename(val_loader.dataset.data[idx]['img_path']) for idx in
                    range(i * val_loader.batch_size, i * val_loader.batch_size + len(images))
                ]
                all_image_ids.extend(image_ids)
                all_labels.extend(labels.numpy())
            else:
                # dual images case
                for k in range(2):
                    all_preds.extend(preds.cpu().numpy())
                    image_ids = [
                        os.path.basename(val_loader.dataset.data[idx][f'img_path{k + 1}']) for idx in
                        range(i * val_loader.batch_size, i * val_loader.batch_size + len(images[k]))
                    ]
                    all_image_ids.extend(image_ids)
                    all_labels.extend(labels.numpy())

            pbar.update(1)

    metrics = compute_metrics(all_preds, all_labels)
    return metrics, val_loss


def test_ensemble(models, test_loader, device, ensemble_method, weights=None, prediction_path='./test_predictions.csv'):

    all_preds = []
    all_image_ids = []

    with tqdm(total=len(test_loader), desc=f'Evaluating', unit=' batch', file=sys.stdout) as pbar:
        for i, data in enumerate(test_loader):
            images = data

            if not isinstance(images, list):
                images = images.to(device)  # single image case
            else:
                images = [x.to(device) for x in images]  # dual images case

            with torch.no_grad():
                if ensemble_method == 'weighted_average':
                    outputs = weighted_average_ensemble(models, images, weights)
                    preds = torch.argmax(outputs, dim=1)
                elif ensemble_method == 'max_voting':
                    preds = max_voting_ensemble(models, images)
                elif ensemble_method == 'bagging':
                    outputs = bagging_ensemble(models, images)
                    preds = torch.argmax(outputs, dim=1)
                else:
                    raise ValueError("Invalid ensemble method")

            if not isinstance(images, list):
                # single image case
                all_preds.extend(preds.cpu().numpy())
                image_ids = [
                    os.path.basename(test_loader.dataset.data[idx]['img_path']) for idx in
                    range(i * test_loader.batch_size, i * test_loader.batch_size + len(images))
                ]
                all_image_ids.extend(image_ids)
            else:
                # dual images case
                for k in range(2):
                    all_preds.extend(preds.cpu().numpy())
                    image_ids = [
                        os.path.basename(test_loader.dataset.data[idx][f'img_path{k + 1}']) for idx in
                        range(i * test_loader.batch_size, i * test_loader.batch_size + len(images[k]))
                    ]
                    all_image_ids.extend(image_ids)

            pbar.update(1)

    # Save predictions to csv file for Kaggle online evaluation
    df = pd.DataFrame({
        'ID': all_image_ids,
        'TARGET': all_preds
    })
    df.to_csv(prediction_path, index=False)
    print(f'[Test] Save predictions to {os.path.abspath(prediction_path)}')

In [35]:
print(f'Evaluating Weighted Average Ensemble')
val_metrics, val_loss = evaluate_ensemble(criterion, models, val_loader, device, 'weighted_average', weights)
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')

print(f'Evaluating Max Voting Ensemble')
val_metrics, val_loss = evaluate_ensemble(criterion, models, val_loader, device, 'max_voting')
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')

# Example usage for testing
print(f'Testing Weighted Average Ensemble')
test_ensemble(models, test_loader, device, 'weighted_average', weights, prediction_path='/content/drive/MyDrive/DL_Project/part_d/test_predictions_weighted_average.csv')

print(f'Testing Max Voting Ensemble')
test_ensemble(models, test_loader, device, 'max_voting', prediction_path='/content/drive/MyDrive/DL_Project/part_d/test_predictions_max_voting.csv')


Evaluating Weighted Average Ensemble
Evaluating: 100%|██████████| 9/9 [00:09<00:00,  1.11s/ batch]
[Val] Kappa: 0.8540 Accuracy: 0.7125 Precision: 0.6919 Recall: 0.7125
Evaluating Max Voting Ensemble
Evaluating: 100%|██████████| 9/9 [00:12<00:00,  1.43s/ batch]
[Val] Kappa: 0.8595 Accuracy: 0.7100 Precision: 0.6981 Recall: 0.7100
Testing Weighted Average Ensemble
Evaluating: 100%|██████████| 9/9 [00:07<00:00,  1.17 batch/s]
[Test] Save predictions to /content/drive/MyDrive/DL_Project/part_d/test_predictions_weighted_average.csv
Testing Max Voting Ensemble
Evaluating: 100%|██████████| 9/9 [00:07<00:00,  1.13 batch/s]
[Test] Save predictions to /content/drive/MyDrive/DL_Project/part_d/test_predictions_max_voting.csv


# Stacking, Boosting

In [36]:

def stacking_boosting_ensemble(models, train_loader, val_loader, device):
    # Collect predictions from base models
    train_preds = []
    train_labels = []
    val_preds = []
    val_labels = []

    for model_index, model in enumerate(models):
        preds = []
        # Collect training predictions
        with torch.no_grad():
            for images, labels in train_loader:
                images = images.to(device)
                outputs = model(images)
                probabilities = torch.nn.functional.softmax(outputs, dim=1).cpu().numpy()
                preds.extend(probabilities)
                # Collect labels only once
                if model_index == 0:
                    train_labels.extend(labels.cpu().numpy())
        train_preds.append(np.vstack(preds))

        preds = []
        # Collect validation predictions
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                probabilities = torch.nn.functional.softmax(outputs, dim=1).cpu().numpy()
                preds.extend(probabilities)
                # Collect labels only once
                if model_index == 0:
                    val_labels.extend(labels.cpu().numpy())
        val_preds.append(np.vstack(preds))

    train_preds = np.concatenate(train_preds, axis=1)
    val_preds = np.concatenate(val_preds, axis=1)
    train_labels = np.array(train_labels)
    val_labels = np.array(val_labels)

    # Train meta-model
    meta_model = LogisticRegression()
    meta_model.fit(train_preds, train_labels)

    # Evaluate meta-model
    val_meta_preds = meta_model.predict(val_preds)
    val_metrics = compute_metrics(val_meta_preds, val_labels)
    val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
    print(f'Stacking Ensemble Metrics:')
    print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')

    # Train boosting model
    base_estimator = DecisionTreeClassifier(max_depth=5)
    boosting_model = AdaBoostClassifier(estimator=base_estimator, n_estimators=50)
    boosting_model.fit(train_preds, train_labels)

    # Evaluate boosting model
    val_boosting_preds = boosting_model.predict(val_preds)
    val_metrics = compute_metrics(val_boosting_preds, val_labels)
    val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
    print(f'Boosting Ensemble Metrics:')
    print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')

    return meta_model, boosting_model

def test_stacking_boosting_ensemble(models, meta_model, boosting_model, test_loader, device, stacking_path, boosting_path):

    all_preds_stacking = []
    all_preds_boosting = []
    all_image_ids = []

    with tqdm(total=len(test_loader), desc='Testing Stacking & Boosting', unit='batch', file=sys.stdout) as pbar:
        for i, data in enumerate(test_loader):
            images = data

            if not isinstance(images, list):
                images = images.to(device)  # single image case
            else:
                images = [x.to(device) for x in images]  # dual images case

            # Store test predictions
            test_preds = []
            for model in models:
                with torch.no_grad():
                    outputs = model(images)
                    probabilities = torch.nn.functional.softmax(outputs, dim=1).cpu().numpy()
                    test_preds.append(probabilities)  # Collect predictions from each model

            # Stack predictions for meta-models (convert list of arrays into single array)
            test_preds = np.hstack(test_preds)

            # Stacking predictions
            preds_stacking = meta_model.predict(test_preds)

            # Boosting predictions
            preds_boosting = boosting_model.predict(test_preds)

            if not isinstance(images, list):
                # single image case
                image_ids = [
                    os.path.basename(test_loader.dataset.data[idx]['img_path']) for idx in
                    range(i * test_loader.batch_size, i * test_loader.batch_size + len(images))
                ]
                all_image_ids.extend(image_ids)
            else:
                # dual images case
                for k in range(2):
                    image_ids = [
                        os.path.basename(test_loader.dataset.data[idx][f'img_path{k + 1}']) for idx in
                        range(i * test_loader.batch_size, i * test_loader.batch_size + len(images[k]))
                    ]
                    all_image_ids.extend(image_ids)

            all_preds_stacking.extend(preds_stacking)
            all_preds_boosting.extend(preds_boosting)

            pbar.update(1)

    # Save stacking predictions to CSV
    df_stacking = pd.DataFrame({
        'ID': all_image_ids,
        'TARGET': all_preds_stacking
    })
    df_stacking.to_csv(stacking_path, index=False)
    print(f"[Test] Stacking predictions saved to {os.path.abspath(stacking_path)}")

    # Save boosting predictions to CSV
    df_boosting = pd.DataFrame({
        'ID': all_image_ids,
        'TARGET': all_preds_boosting
    })
    df_boosting.to_csv(boosting_path, index=False)
    print(f"[Test] Boosting predictions saved to {os.path.abspath(boosting_path)}")

In [37]:
# Evaluate Stacking Ensemble
meta_model, boosting_model = stacking_boosting_ensemble(models, train_loader, val_loader, device)

Stacking Ensemble Metrics:
[Val] Kappa: 0.8554 Accuracy: 0.6875 Precision: 0.6758 Recall: 0.6875
Boosting Ensemble Metrics:
[Val] Kappa: 0.8413 Accuracy: 0.6775 Precision: 0.6699 Recall: 0.6775


In [38]:
stacking_prediction_path = "/content/drive/MyDrive/DL_Project/part_d/test_predictions_stacking.csv"
boosting_prediction_path = "/content/drive/MyDrive/DL_Project/part_d/test_predictions_boosting.csv"
# Invoke function
test_stacking_boosting_ensemble(
models=models,
    meta_model=meta_model,
    boosting_model=boosting_model,
    test_loader=test_loader,
    device=device,
    stacking_path=stacking_prediction_path,
    boosting_path=boosting_prediction_path
)

Testing Stacking & Boosting: 100%|██████████| 9/9 [00:07<00:00,  1.15batch/s]
[Test] Stacking predictions saved to /content/drive/MyDrive/DL_Project/part_d/test_predictions_stacking.csv
[Test] Boosting predictions saved to /content/drive/MyDrive/DL_Project/part_d/test_predictions_boosting.csv


# Bagging

In [46]:
class SingleModels(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = models.efficientnet_b0(pretrained=False)
        state_dict = torch.load('/content/drive/MyDrive/DL_Project/part_b/pretrained/efficientnet_b0.pth', map_location='cpu')
        self.backbone.load_state_dict(state_dict, strict=False)
        num_ftrs = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        return x


In [47]:
from sklearn.utils import resample
from torch.utils.data import DataLoader
import numpy as np
from torchvision import models, transforms

# Prepare data
train_data = pd.read_csv('/content/drive/MyDrive/DL_Project/DeepDRiD/train.csv')

# Parameters
n_bagging_models = 5  # Number of bagging models
bagging_predictions = []

# Metrics storage
bagging_models = []

for bag in range(n_bagging_models):
    print(f"Training Bagging Model {bag + 1}/{n_bagging_models}")

    # Bootstrap sampling
    bootstrap_indices = resample(train_data.index, replace=True, n_samples=len(train_data))
    bootstrap_data = train_data.iloc[bootstrap_indices].reset_index(drop=True)
    bootstrap_data.to_csv(f'/tmp/train_bag{bag + 1}.csv', index=False)

    # Create dataset and data loader
    train_dataset = RetinopathyDataset(
        ann_file=f'/tmp/train_bag{bag + 1}.csv',
        image_dir='/content/drive/MyDrive/DL_Project/DeepDRiD/train/',
        transform=transform_train,
        mode=mode
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Model initialization
    model = SingleModels()
    model = model.to(device)

    # Optimizer and scheduler
    optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    # Train the model
    model, training_loss, validation_loss, training_accuracy, validation_accuracy, train_kappa, val_kappa_list = train_model(
        model, train_loader, val_loader, device, criterion, optimizer,
        lr_scheduler=lr_scheduler, num_epochs=num_epochs,
        checkpoint_path=f'/content/drive/MyDrive/DL_Project/part_d/efficientnet_b0_bagging_model{bag + 1}.pth'
    )
    bagging_models.append(model)

Training Bagging Model 1/5


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-46-607aff457747>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions


Epoch 1/20
Training: 100%|██████████| 25/25 [00:19<00:00,  1.31 batch/s, lr=1.0e-04, Loss=1.2432]
[Train] Kappa: 0.5738 Accuracy: 0.3842 Precision: 0.3853 Recall: 0.3842 Loss: 1.4598
[Train] Class 0: Precision: 0.5358, Recall: 0.6552
[Train] Class 1: Precision: 0.3290, Recall: 0.2857
[Train] Class 2: Precision: 0.1871, Recall: 0.1272
[Train] Class 3: Precision: 0.5068, Recall: 0.3378
[Train] Class 4: Precision: 0.1659, Recall: 0.3178
Evaluating: 100%|██████████| 9/9 [00:04<00:00,  2.21 batch/s]
[Val] Kappa: 0.7465 Accuracy: 0.5475 Precision: 0.5409 Recall: 0.5475

Epoch 2/20
Training: 100%|██████████| 25/25 [00:18<00:00,  1.32 batch/s, lr=1.0e-04, Loss=0.8624]
[Train] Kappa: 0.7918 Accuracy: 0.6308 Precision: 0.6050 Recall: 0.6308 Loss: 1.0610
[Train] Class 0: Precision: 0.6790, Recall: 0.9708
[Train] Class 1: Precision: 0.5990, Recall: 0.4662
[Train] Class 2: Precision: 0.5772, Recall: 0.3114
[Train] Class 3: Precision: 0.6059, Recall: 0.8378
[Train] Class 4: Precision: 0.4167, Recal

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-46-607aff457747>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions


Epoch 1/20
Training: 100%|██████████| 25/25 [00:18<00:00,  1.38 batch/s, lr=1.0e-04, Loss=1.2474]
[Train] Kappa: 0.6082 Accuracy: 0.5142 Precision: 0.4941 Recall: 0.5142 Loss: 1.3573
[Train] Class 0: Precision: 0.6325, Recall: 0.8161
[Train] Class 1: Precision: 0.4513, Recall: 0.4163
[Train] Class 2: Precision: 0.4038, Recall: 0.3320
[Train] Class 3: Precision: 0.5742, Recall: 0.5430
[Train] Class 4: Precision: 0.2427, Recall: 0.1969
Evaluating: 100%|██████████| 9/9 [00:03<00:00,  2.34 batch/s]
[Val] Kappa: 0.7897 Accuracy: 0.6425 Precision: 0.6686 Recall: 0.6425

Epoch 2/20
Training: 100%|██████████| 25/25 [00:17<00:00,  1.40 batch/s, lr=1.0e-04, Loss=0.7817]
[Train] Kappa: 0.8150 Accuracy: 0.6817 Precision: 0.6373 Recall: 0.6817 Loss: 0.9850
[Train] Class 0: Precision: 0.7877, Recall: 0.9598
[Train] Class 1: Precision: 0.6927, Recall: 0.6163
[Train] Class 2: Precision: 0.6154, Recall: 0.5560
[Train] Class 3: Precision: 0.5875, Recall: 0.8507
[Train] Class 4: Precision: 0.2500, Recal

...


Epoch 1/20
Training: 100%|██████████| 25/25 [00:17<00:00,  1.42 batch/s, lr=1.0e-04, Loss=1.1968]
[Train] Kappa: 0.5729 Accuracy: 0.4608 Precision: 0.4601 Recall: 0.4608 Loss: 1.3941
[Train] Class 0: Precision: 0.5156, Recall: 0.8820
[Train] Class 1: Precision: 0.3372, Recall: 0.2377
[Train] Class 2: Precision: 0.5806, Recall: 0.0783
[Train] Class 3: Precision: 0.4642, Recall: 0.5279
[Train] Class 4: Precision: 0.3252, Recall: 0.2920
Evaluating: 100%|██████████| 9/9 [00:03<00:00,  2.36 batch/s]
[Val] Kappa: 0.7730 Accuracy: 0.5825 Precision: 0.5695 Recall: 0.5825

Epoch 2/20
Training: 100%|██████████| 25/25 [00:18<00:00,  1.36 batch/s, lr=1.0e-04, Loss=0.7907]
[Train] Kappa: 0.8098 Accuracy: 0.6458 Precision: 0.6281 Recall: 0.6458 Loss: 1.0160
[Train] Class 0: Precision: 0.7271, Recall: 0.9803
[Train] Class 1: Precision: 0.6489, Recall: 0.5000
[Train] Class 2: Precision: 0.6164, Recall: 0.4261
[Train] Class 3: Precision: 0.5601, Recall: 0.8197
[Train] Class 4: Precision: 0.4688, Recal

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-46-607aff457747>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions


Epoch 1/20
Training: 100%|██████████| 25/25 [00:18<00:00,  1.37 batch/s, lr=1.0e-04, Loss=1.2423]
[Train] Kappa: 0.6468 Accuracy: 0.5408 Precision: 0.5287 Recall: 0.5408 Loss: 1.3419
[Train] Class 0: Precision: 0.5540, Recall: 0.9136
[Train] Class 1: Precision: 0.5472, Recall: 0.1245
[Train] Class 2: Precision: 0.4867, Recall: 0.3244
[Train] Class 3: Precision: 0.5447, Recall: 0.7500
[Train] Class 4: Precision: 0.4500, Recall: 0.0833
Evaluating: 100%|██████████| 9/9 [00:04<00:00,  2.23 batch/s]
[Val] Kappa: 0.7992 Accuracy: 0.6325 Precision: 0.6739 Recall: 0.6325

Epoch 2/20
Training: 100%|██████████| 25/25 [00:17<00:00,  1.40 batch/s, lr=1.0e-04, Loss=0.9374]
[Train] Kappa: 0.8062 Accuracy: 0.6783 Precision: 0.6826 Recall: 0.6783 Loss: 0.9479
[Train] Class 0: Precision: 0.7576, Recall: 0.9738
[Train] Class 1: Precision: 0.7292, Recall: 0.6009
[Train] Class 2: Precision: 0.5887, Recall: 0.3689
[Train] Class 3: Precision: 0.5806, Recall: 0.8571
[Train] Class 4: Precision: 0.7500, Recal

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
<ipython-input-46-607aff457747>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions


Epoch 1/20
Training: 100%|██████████| 25/25 [00:17<00:00,  1.39 batch/s, lr=1.0e-04, Loss=1.1637]
[Train] Kappa: 0.6221 Accuracy: 0.4825 Precision: 0.4510 Recall: 0.4825 Loss: 1.3698
[Train] Class 0: Precision: 0.5297, Recall: 0.9116
[Train] Class 1: Precision: 0.3692, Recall: 0.1026
[Train] Class 2: Precision: 0.4808, Recall: 0.5040
[Train] Class 3: Precision: 0.4219, Recall: 0.2348
[Train] Class 4: Precision: 0.3710, Recall: 0.3651
Evaluating: 100%|██████████| 9/9 [00:03<00:00,  2.29 batch/s]
[Val] Kappa: 0.7915 Accuracy: 0.6250 Precision: 0.6160 Recall: 0.6250

Epoch 2/20
Training: 100%|██████████| 25/25 [00:18<00:00,  1.35 batch/s, lr=1.0e-04, Loss=0.9221]
[Train] Kappa: 0.8215 Accuracy: 0.6675 Precision: 0.6519 Recall: 0.6675 Loss: 0.9989
[Train] Class 0: Precision: 0.7712, Recall: 0.9779
[Train] Class 1: Precision: 0.6417, Recall: 0.5128
[Train] Class 2: Precision: 0.5909, Recall: 0.5242
[Train] Class 3: Precision: 0.6030, Recall: 0.7000
[Train] Class 4: Precision: 0.5373, Recal

In [48]:
print(f'Evaluating Bagging Ensemble')
val_metrics, val_loss = evaluate_ensemble(criterion, bagging_models, val_loader, device, 'bagging')
val_kappa, val_accuracy, val_precision, val_recall = val_metrics[:4]
print(f'[Val] Kappa: {val_kappa:.4f} Accuracy: {val_accuracy:.4f} '
              f'Precision: {val_precision:.4f} Recall: {val_recall:.4f}')

Evaluating Bagging Ensemble
Evaluating: 100%|██████████| 9/9 [00:06<00:00,  1.44 batch/s]
[Val] Kappa: 0.8387 Accuracy: 0.7250 Precision: 0.7114 Recall: 0.7250


In [49]:
print(f'Testing Bagging Ensemble')
test_ensemble(bagging_models, test_loader, device, 'bagging', prediction_path='/content/drive/MyDrive/DL_Project/part_d/test_predictions_bagging.csv')

Testing Bagging Ensemble
Evaluating: 100%|██████████| 9/9 [00:08<00:00,  1.12 batch/s]
[Test] Save predictions to /content/drive/MyDrive/DL_Project/part_d/test_predictions_bagging.csv
